# Phân tích và làm sạch dữ liệu KFC TP.HCM
**DỮ LIỆU MÔ PHỎNG.**
Nguồn: CSV 02 và DULIEUTHO.csv có lỗi, phạm vi 2023–2025. Tệp sạch duy nhất: DataSach/DULIEUSACH.csv.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
project = Path.cwd()
if not (project / 'Script').is_dir():
    project = project.parent if (project.parent / 'Script').is_dir() else next(p for p in project.iterdir() if (p / 'Script').is_dir())
sys.path.insert(0, str(project / 'Script'))
from pipeline import load_clean, build_tables, run
quality = {}
stores, fact = load_clean(project / 'DataGoc', quality)
print(f'{len(stores)} stores, {len(fact)} store-months')
fact.head()

## Làm sạch và truy vết
Chuẩn hóa định dạng, phục hồi đơn/AOV theo công thức cùng dòng, tính lại KPI, loại dòng lỗi/trùng. Nhật ký giữ dòng nguồn và giá trị trước/sau.

In [ ]:
pd.DataFrame(quality['_nhat_ky']).head(20)

## Tính KPI
Tỷ lệ là tỷ số của tổng. Có thể tính YoY 2024/2025 và CAGR 2023–2025; năm đầu để trống kỳ so sánh.

In [ ]:
tables = build_tables(stores, fact)
tables['KPI_Nam'][['nam', 'DT_tong', 'LN_van_hanh_mo_hinh', 'bien_LN_van_hanh', 'CH_binh_quan', 'DT_CH', 'AOV_tong', 'ty_trong_DT_online', 'YoY_doanh_thu']]

In [ ]:
tables['KPI_Thang'][['ky', 'DT_tong', 'tang_truong_ky_truoc', 'LN_van_hanh_mo_hinh', 'bien_LN_van_hanh']]

In [ ]:
tables['KPI_CuaHang_Nam'][['store_id', 'DT_tong', 'LN_van_hanh_mo_hinh', 'bien_LN_van_hanh']].sort_values('bien_LN_van_hanh')

## Xuất sản phẩm
Ô dưới chạy pipeline và tạo lại dữ liệu sạch, KPI, báo cáo Word/PowerPoint, dashboard HTML. Chỉ các đầu ra được cập nhật; DataGoc giữ nguyên. Sau khi trạng thái DAT, có thể Refresh Power BI.

In [ ]:
from presentation import generate
tables, quality = run(project)
generate(tables, quality, project)
print(quality['trang_thai'])
print(project / 'BaoCao' / 'dashboard.html')

## Giới hạn kết luận
Lợi nhuận là lợi nhuận vận hành mô hình, chưa phải lợi nhuận ròng. Không có thông tin thị trường, khách hàng định danh hoặc quận/huyện. Không suy rộng kết quả mô phỏng thành đánh giá kinh doanh thực tế của KFC.